In [0]:
%pip install faker

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.5 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ATL-02: Generate synthetic data and load to bronze as Delta tables
from pyspark.sql import functions as F
from faker import Faker
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

catalog = "atlas"

# ---------- MEMBERS (~10k) ----------
regions = ["Madrid", "Catalonia", "Andalusia", "Valencia", "Basque Country"]
members = [(
    i,                                   # member_id
    fake.unique.ssn(),                   # national_id
    fake.date_of_birth(minimum_age=0, maximum_age=95).isoformat(),  # birth_date
    random.choice(["M", "F"]),           # gender
    random.choice(regions)               # region
) for i in range(1, 10001)]
members_df = spark.createDataFrame(members, ["member_id","national_id","birth_date","gender","region"])
members_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.members")

# ---------- PROVIDERS (~500) ----------
specialties = ["Cardiology","Oncology","Pediatrics","Radiology","General"]
providers = [(
    i, fake.company(), random.choice(specialties), random.choice(regions)
) for i in range(1, 501)]
providers_df = spark.createDataFrame(providers, ["provider_id","name","specialty","region"])
providers_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.providers")

# ---------- CLAIMS (~100k) with deliberate problems ----------
valid_icd10 = ["I10","E11","J45","M54","K21"]
invalid_icd10 = ["XX99","0000","ABCD"]
claims = []
for i in range(1, 100001):
    service = fake.date_between(start_date="-1y", end_date="today")
    late = random.random() < 0.05            # ~5% late billing
    billing = fake.date_between(start_date=service, end_date="+120d") if late else service
    bad_code = random.random() < 0.04        # ~4% invalid ICD-10
    icd10 = random.choice(invalid_icd10) if bad_code else random.choice(valid_icd10)
    neg = random.random() < 0.02             # ~2% negative amounts
    amount = round(random.uniform(-500, 5000), 2) if neg else round(random.uniform(10, 5000), 2)
    claims.append((i, random.randint(1,10000), random.randint(1,500),
                   service.isoformat(), billing.isoformat(), icd10, amount, "submitted"))

# ~3% duplicates/resubmissions
dupes = random.sample(claims, int(len(claims)*0.03))
claims.extend([(c[0],)+c[1:] for c in dupes])

claims_df = spark.createDataFrame(claims,
    ["claim_id","member_id","provider_id","service_date","billing_date","icd10_code","amount","status"])
claims_df.write.mode("overwrite").saveAsTable(f"{catalog}.bronze.claims")

In [0]:
# ---------- Evidence: row counts ----------
for t in ["members","providers","claims"]:
    print(t, spark.table(f"{catalog}.bronze.{t}").count())

members 10000
providers 500
claims 103000
